In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
 
warnings.filterwarnings("ignore")
REPS = int(os.environ.get("REPS", 10))
B = int(os.environ.get("B_BOOT", 200))
KIND = os.environ.get("BIN_MODEL", "xgb")
FS = os.environ.get("BIN_FS", "full")
KS = [1, 2, 3, 4, 5]
os.makedirs("results", exist_ok=True)
 
df = pd.read_csv("data/processed/features.csv")
g1 = pd.read_csv("results/feature_groups.csv", index_col=0)["group"]
COLS = g1.index.tolist() if FS == "full" else \
    [c for c in g1.index if g1[c] != "dynamics"]
tr = df[df.split == "train"].reset_index(drop=True)
va = df[df.split == "val"].reset_index(drop=True)
E = df[df.split.isin(["val", "test"])].reset_index(drop=True)
codes, _ = pd.factorize(E.company_name)
nU = codes.max() + 1
W = np.random.default_rng(42).multinomial(
    nU, np.full(nU, 1 / nU), size=B).astype(float)
 
 
def rho(a, b):
    return float(spearmanr(a, b)[0])
 
 
def fit_bin(kind, Xtr, ytr, Xva, yva, seed):
    if kind == "xgb":
        from xgboost import XGBClassifier
        spw = float((ytr == 0).sum() / max((ytr == 1).sum(), 1))
        m = XGBClassifier(n_estimators=800, learning_rate=0.03, max_depth=4,
                          subsample=0.8, colsample_bytree=0.8,
                          min_child_weight=5, reg_lambda=2.0,
                          scale_pos_weight=spw, eval_metric="aucpr",
                          early_stopping_rounds=50, tree_method="hist",
                          n_jobs=-1, random_state=seed)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
        return m
    r = np.random.default_rng(seed)
    keep = r.random(len(ytr)) < 0.85
    Xs, ys = Xtr[keep], ytr[keep]
    if kind == "ebm":
        from interpret.glassbox import ExplainableBoostingClassifier
        w = np.where(ys == 1, (ys == 0).sum() / max((ys == 1).sum(), 1), 1.0)
        m = ExplainableBoostingClassifier(random_state=seed)
        m.fit(Xs, ys, sample_weight=w)
        return m
    m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                      LogisticRegression(C=0.1, class_weight="balanced",
                                         max_iter=3000))
    m.fit(Xs, ys)
    return m
 
 
def contrib(kind, model, X):
    if kind == "logreg":
        imp, sc, lr = (s[1] for s in model.steps)
        return sc.transform(imp.transform(X)) * lr.coef_[0]
    if kind == "xgb":
        import xgboost as xgb
        try:
            ir = (0, int(model.best_iteration) + 1)
        except Exception:
            ir = (0, 0)
        c = model.get_booster().predict(xgb.DMatrix(X, missing=np.nan),
                                        pred_contribs=True,
                                        iteration_range=ir)
        return c[:, :-1]
    T = model.eval_terms(X)
    out = np.zeros((len(X), X.shape[1]))
    for j, feats in enumerate(model.term_features_):
        for f in feats:
            out[:, f] += T[:, j] / len(feats)
    return out
 
 
def fit_k(k, d_tr, seed):
    mt = ((d_tr.failed_co == 0) | (d_tr.yrs_before_fail == k)).values
    mv = ((va.failed_co == 0) | (va.yrs_before_fail == k)).values
    ytr = (d_tr.yrs_before_fail[mt] == k).astype(int).values
    yva = (va.yrs_before_fail[mv] == k).astype(int).values
    return fit_bin(KIND, d_tr.loc[mt, COLS], ytr, va.loc[mv, COLS], yva, seed)
 
 
def pos_imp(model, k):
    X = E.loc[(E.yrs_before_fail == k).values, COLS]
    return np.abs(contrib(KIND, model, X)).mean(0)
 
 
# ------------------------------- A. random-half control (by company)
companies = tr.company_name.unique()
P1_n = tr[tr.year <= 2005].shape[0]
rows = []
for k in KS:
    rhos = []
    for r in range(REPS):
        rr = np.random.default_rng(1000 + r)
        perm = rr.permutation(companies)
        half = set(perm[: len(perm) // 2])
        isA = tr.company_name.isin(half)
        A, Bh = tr[isA].reset_index(drop=True), tr[~isA].reset_index(drop=True)
        if (A.yrs_before_fail == k).sum() < 10 or \
                (Bh.yrs_before_fail == k).sum() < 10:
            continue
        rhos.append(rho(pos_imp(fit_k(k, A, r), k),
                        pos_imp(fit_k(k, Bh, r), k)))
    if not rhos:
        print(f"  control k={k}: too few positives, skipped")
        continue
    rows.append({"k": k, "random_half_spearman_mean": np.mean(rhos),
                 "random_half_spearman_min": np.min(rhos),
                 "random_half_spearman_max": np.max(rhos),
                 "n_reps": len(rhos)})
    print(f"  control k={k} done")
ctl = pd.DataFrame(rows)
if os.path.exists("results/step6_period_retrain.csv"):
    p = pd.read_csv("results/step6_period_retrain.csv")[
        ["k", "spearman_P1_vs_P2"]]
    ctl = ctl.merge(p, on="k", how="left")
    ctl["period_minus_random"] = (ctl.spearman_P1_vs_P2
                                  - ctl.random_half_spearman_mean)
ctl.to_csv("results/step7_period_control.csv", index=False)
 
# ------------------------- B. CIs for lead-time AUCs (seed 0, full train)
rows = []
for k in KS:
    m = fit_k(k, tr, 0)
    me = ((E.failed_co == 0) | (E.yrs_before_fail == k)).values
    ye = (E.yrs_before_fail[me] == k).astype(int).values
    s = m.predict_proba(E.loc[me, COLS])[:, 1]
    cu = codes[me]
    roc, pr = roc_auc_score(ye, s), average_precision_score(ye, s)
    bs = []
    for b in range(B):
        w = W[b][cu]
        if (w[ye == 1] > 0).any():
            bs.append((roc_auc_score(ye, s, sample_weight=w),
                       average_precision_score(ye, s, sample_weight=w)))
    bs = np.array(bs)
    prev = ye.mean()
    rows.append({"k_years_before": k, "heldout_pos": int(ye.sum()),
                 "ROC_AUC": roc,
                 "ROC_lo": np.percentile(bs[:, 0], 2.5),
                 "ROC_hi": np.percentile(bs[:, 0], 97.5),
                 "PR_AUC": pr, "PR_lo": np.percentile(bs[:, 1], 2.5),
                 "PR_hi": np.percentile(bs[:, 1], 97.5),
                 "PR_lift": pr / prev})
auc = pd.DataFrame(rows)
auc.to_csv("results/step7_bin_auc_ci.csv", index=False)
 
pd.set_option("display.width", 200)
print("\n=== Period split vs random-half control (importance rank corr) ===")
print(ctl.round(3).to_string(index=False))
print("\n=== Lead-time AUCs with 95% company-bootstrap CIs ===")
print(auc.round(3).to_string(index=False))
 

  control k=1 done
  control k=2 done
  control k=3 done
  control k=4 done
  control k=5 done

=== Period split vs random-half control (importance rank corr) ===
 k  random_half_spearman_mean  random_half_spearman_min  random_half_spearman_max  n_reps  spearman_P1_vs_P2  period_minus_random
 1                      0.444                     0.276                     0.556      10              0.255               -0.189
 2                      0.340                     0.126                     0.520      10              0.343                0.004
 3                      0.251                     0.011                     0.357      10              0.227               -0.025
 4                      0.261                    -0.109                     0.421      10              0.104               -0.157
 5                      0.291                     0.050                     0.524      10              0.299                0.008

=== Lead-time AUCs with 95% company-bootstrap CIs ===
 k